<a href="https://colab.research.google.com/github/dasha3000/python-ai-Evdokimova-Daria/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных о памятниках с помощью pandas.

**Данные:**
- [`monuments_country_style.csv`](https://github.com/dasha3000/python-ai-Evdokimova-Daria/blob/main/data/monuments_country_style.csv) — информация о памятниках: метка, страна, стиль

**Структура файла:**
| Столбец | Описание |
|---------|----------|
| `monument` | URI объекта на Wikidata |
| `monumentLabel` | Название памятника |
| `countryLabel` | Страна расположения |
| `styleLabel` | Художественный стиль |

**Что мы делаем:**
1. Клонируем ваш репозиторий GitHub в Colab
2. Читаем CSV-файл `monuments_country_style.csv` в pandas DataFrame
3. Очищаем и переименовываем столбцы при необходимости
4. Смотрим структуру данных и делаем быструю валидацию

In [12]:
# Шаг 0.

import os
import pandas as pd

## 🐱 [1] Клонируем репозиторий курса в Colab

In [13]:
# 🐱 Шаг 1. Клонируем ваш репозиторий курса в Colab


repo = "python-ai-Evdokimova-Daria"  # ← ИЗМЕНЕНО: имя вашего репозитория
repo_path = f"/content/{repo}"  # абсолютный путь — не зависит от cwd

if not os.path.exists(repo_path):          # всегда проверяет /content/python-ai-Evdokimova-Daria
    !git clone -q https://github.com/dasha3000/python-ai-Evdokimova-Daria.git  # ← ИЗМЕНЕНО: URL вашего репозитория

if os.getcwd() != repo_path:               # точное сравнение, не endswith
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Evdokimova-Daria


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

414 строк = N памятников × несколько стилей на каждый (длинный формат)

In [14]:
# 🐱 Шаг 2A. Чтение CSV-файла в pandas + анализ формата данных

import pandas as pd

# 🔹 1) Чтение файла
df_monuments = pd.read_csv("data/monuments_country_style.csv", encoding="utf-8")

# 🔹 2) Базовая статистика
print("✅ Загружено строк в df_monuments:", len(df_monuments))

# 🔹 3) Фиксация длинного формата
# ⚠️ ВНИМАНИЕ: на этом этапе столбец с URL ещё называется 'monument'!
# Переименование в 'URL' произойдёт в Шаге 2B (очистка).

url_column = 'monument' if 'monument' in df_monuments.columns else 'URL'

n_unique = df_monuments[url_column].nunique()
avg_styles = len(df_monuments) / n_unique

print(f"🏛️ Уникальных памятников (по столбцу '{url_column}'): {n_unique}")
print(f"📐 Среднее число стилей на памятник: {avg_styles:.1f}")

if avg_styles > 1.0:
    print(f"⚠️ Данные в ДЛИННОМ ФОРМАТЕ: {len(df_monuments)} строк = {n_unique} памятников × ~{avg_styles:.1f} стилей")
    print("💡 При анализе используйте .nunique() для подсчёта памятников, а не len(df)")
else:
    print("✅ Каждый памятник имеет ровно один стиль — формат простой")

# 🔹 4) Быстрая проверка структуры
print("\n📋 Первые 3 строки:")
display(df_monuments.head(3))

print("\n📊 Информация о столбцах:")
print(df_monuments.info())

# 🔹 5) Бонус: проверка на дубликаты (памятник + стиль)
print(f"\n🔍 Проверка на дубликаты ({url_column} + style):")
duplicates = df_monuments.duplicated(subset=[url_column, 'styleLabel' if 'styleLabel' in df_monuments.columns else 'style']).sum()
print(f"Дубликатов пар (памятник, стиль): {duplicates}")
if duplicates > 0:
    print("⚠️ Возможно, есть повторяющиеся записи — проверьте данные")

# 🔹 6) Подсказка для следующего шага
print(f"\n📌 Следующий шаг (2B): переименуем '{url_column}' → 'URL' и уберём суффиксы *Label")

✅ Загружено строк в df_monuments: 414
🏛️ Уникальных памятников (по столбцу 'monument'): 337
📐 Среднее число стилей на памятник: 1.2
⚠️ Данные в ДЛИННОМ ФОРМАТЕ: 414 строк = 337 памятников × ~1.2 стилей
💡 При анализе используйте .nunique() для подсчёта памятников, а не len(df)

📋 Первые 3 строки:


,monument,monumentLabel,countryLabel,styleLabel
0,http://www.wikidata.org/entity/Q3852976,Monument to cardinal De Braye,Италия,готическая скульптура
1,http://www.wikidata.org/entity/Q3323370,Héloïse and Abélard's tomb,Франция,неоготика
2,http://www.wikidata.org/entity/Q11363889,Chūson-ji Konjikidō,Япония,Pure Land worship



📊 Информация о столбцах:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 414 entries, 0 to 413
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   monument       414 non-null    object
 1   monumentLabel  414 non-null    object
 2   countryLabel   414 non-null    object
 3   styleLabel     414 non-null    object
dtypes: object(4)
memory usage: 13.1+ KB
None

🔍 Проверка на дубликаты (monument + style):
Дубликатов пар (памятник, стиль): 56
⚠️ Возможно, есть повторяющиеся записи — проверьте данные

📌 Следующий шаг (2B): переименуем 'monument' → 'URL' и уберём суффиксы *Label


## 🧹 [2B] Очистка и переименование столбцов

В исходном CSV-файле есть **технические столбцы**, которые полезны для Викиданных, но мешают простому анализу:

- Столбец `monument` с URL (ссылкой на объект Wikidata) — **переименуем в `URL`**, чтобы сохранить возможность перехода к оригинальной записи.
- Столбцы `monumentLabel`, `countryLabel`, `styleLabel` содержат читаемые подписи (названия).

В этом шаге мы:
- переименуем столбец с URL Wikidata (`monument` → `URL`);
- переименуем `monumentLabel → monument`, `countryLabel → country`, `styleLabel → style` (уберём суффикс `Label`);
- проверим данные на пропущенные значения.

> ⚠️ **Важно:** если в ваших данных есть столбцы с URL Wikidata и столбцы вида `*Label`, этот шаг обязателен, чтобы получить аккуратные таблички для анализа. Столбец `URL` пригодится, если нужно будет быстро перейти к оригинальной записи в Викиданных.

In [15]:
# 🧹 Шаг 2B. Очистка и переименование столбцов для df_monuments

# Проверка: нужна ли очистка?
if "monumentLabel" in df_monuments.columns:

    # Переименовываем столбцы:
    # 1) URL-столбец: monument → URL
    # 2) Label-столбцы: убираем суффикс Label
    df_monuments = df_monuments.rename(columns={
        "monument": "URL",              # ← ИЗМЕНЕНО: технический URL → удобное имя
        "monumentLabel": "monument",    # ← ИЗМЕНЕНО: ваши столбцы
        "countryLabel": "country",
        "styleLabel": "style",
    })
    print("✅ Столбцы переименованы: monument→URL, *Label→без суффикса")

    # Проверка на пропущенные значения
    print("\n📊 Пропущенные значения в столбцах:")
    print(df_monuments.isnull().sum())

else:
    print("⏭️ df_monuments уже очищен, пропускаем")

# 🔍 Быстрая валидация результата
print("\n✅ Данные готовы к анализу")
print("\n📋 Первые 5 строк очищенного DataFrame:")
display(df_monuments.head())

✅ Столбцы переименованы: monument→URL, *Label→без суффикса

📊 Пропущенные значения в столбцах:
URL         0
monument    0
country     0
style       0
dtype: int64

✅ Данные готовы к анализу

📋 Первые 5 строк очищенного DataFrame:


,URL,monument,country,style
0,http://www.wikidata.org/entity/Q3852976,Monument to cardinal De Braye,Италия,готическая скульптура
1,http://www.wikidata.org/entity/Q3323370,Héloïse and Abélard's tomb,Франция,неоготика
2,http://www.wikidata.org/entity/Q11363889,Chūson-ji Konjikidō,Япония,Pure Land worship
3,http://www.wikidata.org/entity/Q20875264,Panteó Mundet,Испания,каталонский модерн
4,http://www.wikidata.org/entity/Q20875307,Sepulcre de la família Majó,Испания,неоклассицизм


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор DataFrame с данными о памятниках:

- посмотрим размер таблицы (`shape`);
- выведём список столбцов;
- посмотрим первые несколько строк;
- проверим типы данных в столбцах.

**Структура данных после очистки:**
| Столбец | Описание | Тип данных |
|---------|----------|------------|
| `URL` | Ссылка на объект Wikidata | строка |
| `monument` | Название памятника | строка |
| `country` | Страна расположения | строка |
| `style` | Художественный стиль | строка |

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы быстро получать информацию о DataFrame.

> ⚠️ **Важно:** в данном наборе данных нет числовых столбцов, поэтому статистика по числовым полям (как `capital_cost` в примере с мультфильмами) не применяется. Вместо этого мы посмотрим на уникальные значения категориальных столбцов (страны, стили).

In [16]:
# 🔍 Шаг 3. Обзор данных

# =============================================================================
# 🔹 0) Универсальные имена столбцов (работает до и после очистки)
# =============================================================================

# Определяем, как сейчас называется столбец с идентификатором памятника
if 'URL' in df_monuments.columns:
    ID_COL = 'URL'
    print("✅ Используем очищенное имя столбца: 'URL'")
elif 'monument' in df_monuments.columns:
    ID_COL = 'monument'
    print("ℹ️ Используем исходное имя столбца: 'monument' (очистка ещё не выполнена)")
else:
    raise KeyError("Не найден столбец с идентификатором памятника!")

# Определяем имя столбца со стилем
STYLE_COL = 'style' if 'style' in df_monuments.columns else 'styleLabel'
COUNTRY_COL = 'country' if 'country' in df_monuments.columns else 'countryLabel'

# =============================================================================
# 🔹 1) Функция быстрого обзора DataFrame
# =============================================================================

def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("=" * 60)
    print(f"📏 Размер таблицы: {df.shape} ({df.shape[0]} строк × {df.shape[1]} столбцов)")
    print(f"📋 Столбцы: {', '.join(df.columns)}")
    print(f"\n📊 Типы данных:")
    print(df.dtypes)
    print(f"\n📋 Первые {n} строк:")
    display(df.head(n))

    # Дополнительно: информация о пропусках
    print(f"\n❓ Пропущенные значения:")
    print(df.isnull().sum())

# =============================================================================
# 🔹 2) Вызов функции для основного DataFrame
# =============================================================================

show_info(df_monuments, "Памятники: страны и стили (df_monuments)")

# =============================================================================
# 🔹 3) Анализ уникальных памятников (с учётом длинного формата)
# =============================================================================

print("\n" + "=" * 60)
print("🏛️ АНАЛИЗ УНИКАЛЬНЫХ ПАМЯТНИКОВ")
print("=" * 60)

# Убираем дубликаты по идентификатору памятника
df_unique = df_monuments.drop_duplicates(subset=ID_COL)

print(f"\n✅ df_unique: {len(df_unique)} уникальных памятников (из {len(df_monuments)} строк)")
print(f"💡 Разница ({len(df_monuments) - len(df_unique)} строк) — это памятники с несколькими стилями")

# Быстрая статистика по уникальным памятникам
print(f"\n📊 Статистика по уникальным памятникам:")
print(f"  • Уникальных стран: {df_unique[COUNTRY_COL].nunique()}")
print(f"  • Уникальных стилей: {df_unique[STYLE_COL].nunique()}")

# Топ стран по количеству уникальных памятников
print(f"\n🌍 Топ-5 стран по числу уникальных памятников:")
print(df_unique[COUNTRY_COL].value_counts().head())

# Топ стилей по количеству уникальных памятников
print(f"\n🎨 Топ-5 стилей по числу уникальных памятников:")
print(df_unique[STYLE_COL].value_counts().head())

# =============================================================================
# 🔹 4) Сравнение: все строки vs уникальные памятники
# =============================================================================

print("\n" + "=" * 60)
print("📈 СРАВНЕНИЕ: ВСЕ ЗАПИСИ vs УНИКАЛЬНЫЕ ПАМЯТНИКИ")
print("=" * 60)

comparison = pd.DataFrame({
    'Метрика': ['Всего строк', 'Уникальных памятников', 'Строк на памятник (среднее)'],
    'Значение': [
        len(df_monuments),
        df_monuments[ID_COL].nunique(),
        f"{len(df_monuments) / df_monuments[ID_COL].nunique():.2f}"
    ]
})
print(comparison.to_string(index=False))

print(f"\n✅ Обзор данных завершён! 🗿✨")

✅ Используем очищенное имя столбца: 'URL'

📊 Памятники: страны и стили (df_monuments)
📏 Размер таблицы: (414, 4) (414 строк × 4 столбцов)
📋 Столбцы: URL, monument, country, style

📊 Типы данных:
URL         object
monument    object
country     object
style       object
dtype: object

📋 Первые 5 строк:


,URL,monument,country,style
0,http://www.wikidata.org/entity/Q3852976,Monument to cardinal De Braye,Италия,готическая скульптура
1,http://www.wikidata.org/entity/Q3323370,Héloïse and Abélard's tomb,Франция,неоготика
2,http://www.wikidata.org/entity/Q11363889,Chūson-ji Konjikidō,Япония,Pure Land worship
3,http://www.wikidata.org/entity/Q20875264,Panteó Mundet,Испания,каталонский модерн
4,http://www.wikidata.org/entity/Q20875307,Sepulcre de la família Majó,Испания,неоклассицизм



❓ Пропущенные значения:
URL         0
monument    0
country     0
style       0
dtype: int64

🏛️ АНАЛИЗ УНИКАЛЬНЫХ ПАМЯТНИКОВ

✅ df_unique: 337 уникальных памятников (из 414 строк)
💡 Разница (77 строк) — это памятники с несколькими стилями

📊 Статистика по уникальным памятникам:
  • Уникальных стран: 40
  • Уникальных стилей: 77

🌍 Топ-5 стран по числу уникальных памятников:
country
Чехия       90
Испания     59
Италия      34
Германия    23
Мексика     19
Name: count, dtype: int64

🎨 Топ-5 стилей по числу уникальных памятников:
style
барокко                     79
модерн                      43
Anti-monuments in Mexico    18
социалистический реализм    13
классицизм                  11
Name: count, dtype: int64

📈 СРАВНЕНИЕ: ВСЕ ЗАПИСИ vs УНИКАЛЬНЫЕ ПАМЯТНИКИ
                    Метрика Значение
                Всего строк      414
      Уникальных памятников      337
Строк на памятник (среднее)     1.23

✅ Обзор данных завершён! 🗿✨


## 🎨 [4] Анализ стилевого разнообразия и «стилей-призраков»

В этом блоке мы исследуем два интересных аспекта данных о памятниках:

### 📊 2. Индекс стилевого разнообразия по странам
Не все страны одинаково «стилистически богаты». Мы посчитаем:
- **Простой индекс**: количество уникальных стилей в каждой стране;
- **Индекс Шеннона**: учитывает не только число стилей, но и равномерность их распределения (высокое значение = стили представлены примерно поровну, низкое = доминирует 1–2 стиля).

> 📌 *Пример*: если в стране 10 памятников и все в стиле «барокко» — разнообразие низкое. Если те же 10 памятников распределены по 5 разным стилям — разнообразие высокое.

### 👻 4. «Стили-призраки»: редкие и уникальные комбинации
Некоторые стили встречаются крайне редко или только в одной стране. Мы найдём:
- **Эксклюзивные стили**: встречаются только в одной стране (например, «Anti-monuments in Mexico»);
- **Редкие стили**: встречаются 1–3 раза во всём датасете;
- **Страны-«хранители»**: какие страны имеют больше всего уникальных стилей.

> 🔍 *Зачем это нужно?* Такие находки помогают обнаружить локальные художественные явления, культурные феномены и потенциальные ошибки в разметке данных.

### 📈 Что мы сделаем:
1. Посчитаем простой индекс и индекс Шеннона для каждой страны;
2. Выведем топ-10 самых «разнообразных» и самых «однородных» стран;
3. Найдём все эксклюзивные и редкие стили;

> ⚠️ **Примечание**: Индекс Шеннона требует установки `scipy`. Если библиотека не установлена, код автоматически использует упрощённую версию.

In [17]:
# 🎨 Шаг 4. Индекс разнообразия и «стили-призраки» + разведка для задания 3

import pandas as pd
import numpy as np
from itertools import combinations
import collections
import re

print("🚀 Запуск анализа стилевого разнообразия...\n")

# =============================================================================
# 🔹 ЧАСТЬ 1: Индекс стилевого разнообразия по странам
# =============================================================================

def calculate_shannon_entropy(counts):
    """
    Рассчитывает индекс Шеннона для серии подсчётов.
    H = -sum(p * log2(p)), где p - доля каждого стиля.
    """
    counts = counts[counts > 0]  # убираем нули
    if len(counts) == 0:
        return 0
    proportions = counts / counts.sum()
    entropy = -np.sum(proportions * np.log2(proportions))
    return entropy

# Группируем по стране и считаем стили
country_style_counts = df_monuments.groupby('country')['style'].value_counts()

# Простой индекс: количество уникальных стилей в стране
simple_diversity = df_monuments.groupby('country')['style'].nunique().rename('unique_styles_count')

# Индекс Шеннона: учитывает равномерность распределения
shannon_diversity = []
for country in df_monuments['country'].unique():
    styles_in_country = country_style_counts.get(country, pd.Series(dtype=int))
    entropy = calculate_shannon_entropy(styles_in_country)
    shannon_diversity.append({'country': country, 'shannon_entropy': entropy})

shannon_diversity_df = pd.DataFrame(shannon_diversity).set_index('country')

# Объединяем метрики
diversity_df = pd.concat([simple_diversity, shannon_diversity_df], axis=1).fillna(0)
diversity_df['total_monuments'] = df_monuments.groupby('country').size()

print("📊 ИНДЕКС СТИЛЕВОГО РАЗНООБРАЗИЯ ПО СТРАНАМ")
print("=" * 70)
print("\n🏆 Топ-10 стран по количеству уникальных стилей:")
display(diversity_df.nlargest(10, 'unique_styles_count')[['unique_styles_count', 'shannon_entropy', 'total_monuments']])

print("\n🧱 Топ-10 самых «однородных» стран (меньше всего стилей):")
display(diversity_df.nsmallest(10, 'unique_styles_count')[['unique_styles_count', 'shannon_entropy', 'total_monuments']])

print("\n📈 Корреляция между числом памятников и разнообразием:")
corr = diversity_df['total_monuments'].corr(diversity_df['unique_styles_count'])
print(f"Коэффициент корреляции Пирсона: {corr:.3f}")
print("💡 Интерпретация: чем ближе к 1, тем сильнее связь «больше памятников → больше стилей»")

# =============================================================================
# 🔹 ЧАСТЬ 2: «Стили-призраки» — редкие и уникальные комбинации
# =============================================================================

print("\n" + "=" * 70)
print("👻 АНАЛИЗ «СТИЛЕЙ-ПРИЗРАКОВ»")
print("=" * 70)

# Сколько стран представлено в каждом стиле
style_country_counts = df_monuments.groupby('style')['country'].nunique()
style_total_counts = df_monuments['style'].value_counts()

# Эксклюзивные стили: встречаются только в ОДНОЙ стране
exclusive_styles = style_country_counts[style_country_counts == 1]
exclusive_df = pd.DataFrame({
    'style': exclusive_styles.index,
    'country': [df_monuments[df_monuments['style'] == s]['country'].iloc[0] for s in exclusive_styles.index],
    'occurrences': [style_total_counts[s] for s in exclusive_styles.index]
}).sort_values('occurrences', ascending=False)

print(f"\n✨ Эксклюзивные стили (встречаются только в одной стране): {len(exclusive_df)}")
print("Топ-15 по частоте:")
display(exclusive_df.head(15))

# Редкие стили: встречаются 1-3 раза во всём датасете
rare_styles = style_total_counts[style_total_counts <= 3]
rare_df = pd.DataFrame({
    'style': rare_styles.index,
    'total_occurrences': rare_styles.values,
    'countries_count': [style_country_counts.get(s, 0) for s in rare_styles.index]
}).sort_values('total_occurrences')

print(f"\n🕯️ Редкие стили (встречаются 1–3 раза): {len(rare_df)}")
print("Полный список:")
display(rare_df)

# Страны-«хранители»: у каких стран больше всего эксклюзивных стилей
exclusive_by_country = exclusive_df.groupby('country').size().rename('exclusive_styles_count')
print(f"\n🏰 Страны с наибольшим числом эксклюзивных стилей:")
display(exclusive_by_country.nlargest(10))

# 🔍 Детальный просмотр: примеры памятников с эксклюзивными стилями
print("\n🔎 Примеры памятников с эксклюзивными стилями:")
sample_exclusive = df_monuments[df_monuments['style'].isin(exclusive_df['style'].head(5).values)]
display(sample_exclusive[['URL', 'monument', 'country', 'style']].head(10))

# =============================================================================
# 🔹 ЧАСТЬ 3: Разведка для задания 3 (ТРИ НОВЫХ БЛОКА)
# =============================================================================

print("\n" + "=" * 70)
print("🔍 РАЗВЕДКА ДЛЯ ЗАДАНИЯ 3: ГРАФИКИ И ВИЗУАЛИЗАЦИЯ")
print("=" * 70)

# ─────────────────────────────────────────────────────────────────────────────
# 📌 БЛОК А. Сколько памятников имеют 2 и более стилей?
# ─────────────────────────────────────────────────────────────────────────────
# ← ВСТАВЛЕНО: сразу после Части 2

print("\n📊 БЛОК А. Распределение числа стилей на памятник")
print("-" * 70)

styles_per_monument = df_monuments.groupby('URL')['style'].count()
print("Распределение числа стилей на памятник:")
print(styles_per_monument.value_counts().sort_index())

multi_style_monuments = (styles_per_monument > 1).sum()
total_monuments = df_monuments['URL'].nunique()

print(f"\n🏛️ Памятников с 2+ стилями: {multi_style_monuments} из {total_monuments}")
print(f"💡 Доля: {multi_style_monuments/total_monuments*100:.1f}%")

if multi_style_monuments > 0:
    print("⚠️ Внимание: есть памятники с несколькими стилями — это важно для визуализации!")
else:
    print("✅ Все памятники имеют ровно один стиль — упрощает построение графиков")

# ─────────────────────────────────────────────────────────────────────────────
# 📌 БЛОК Б. Топ-10 пар стилей, встречающихся вместе
# ─────────────────────────────────────────────────────────────────────────────
# ← ВСТАВЛЕНО: после Блока А

print("\n" + "-" * 70)
print("📊 БЛОК Б. Топ-10 пар стилей, встречающихся в одном памятнике")
print("-" * 70)

pairs = []
for url, group in df_monuments.groupby('URL'):
    styles = group['style'].tolist()
    if len(styles) > 1:
        for a, b in combinations(sorted(styles), 2):
            pairs.append((a, b))

if len(pairs) > 0:
    print("Топ-10 пар стилей, встречающихся в одном памятнике:")
    for pair, cnt in collections.Counter(pairs).most_common(10):
        print(f"  {cnt}x  {pair[0]}  +  {pair[1]}")
    print(f"\n💡 Всего найдено пар стилей: {len(pairs)}")
    print("📈 Это можно визуализировать как сетевой граф связей между стилями!")
else:
    print("⚠️ Пар стилей не найдено — каждый памятник имеет только один стиль")
    print("💡 Для визуализации лучше использовать простые бар-чарты, а не сетевые графы")

# ─────────────────────────────────────────────────────────────────────────────
# 📌 БЛОК В. Доля стилей с не-русскими названиями
# ─────────────────────────────────────────────────────────────────────────────
# ← ВСТАВЛЕНО: после Блока Б

print("\n" + "-" * 70)
print("📊 БЛОК В. Доля стилей с не-русскими названиями (латиница)")
print("-" * 70)

not_ru = df_monuments['style'].dropna().apply(lambda x: bool(re.search(r'[a-zA-Z]', x)))
total_styles = len(df_monuments['style'].dropna())
latin_styles = not_ru.sum()

print(f"Строк со стилем на латинице: {latin_styles} из {total_styles}")
print(f"💡 Доля: {latin_styles/total_styles*100:.1f}%")

if latin_styles > 0:
    print("\n🌐 Какие именно стили (Топ-15):")
    print(df_monuments[not_ru]['style'].value_counts().head(15))
    print("\n⚠️ Внимание: смешанная кодировка названий может повлиять на группировку!")
    print("💡 Рекомендация: для чистоты анализа можно привести все названия к одному языку")
else:
    print("\n✅ Все названия стилей на кириллице — проблем с кодировкой не будет")

# =============================================================================
# 🔹 ИТОГОВЫЙ ВЫВОД
# =============================================================================

print("\n" + "=" * 70)
print("✅ РАЗВЕДКА ДАННЫХ ЗАВЕРШЕНА! 🗿✨")
print("=" * 70)

print("\n📋 Краткое резюме для планирования визуализаций:")
print(f"  • Стран: {df_monuments['country'].nunique()}")
print(f"  • Стилей: {df_monuments['style'].nunique()}")
print(f"  • Памятников с 2+ стилями: {multi_style_monuments}")
print(f"  • Стилей на латинице: {latin_styles}")

print("\n🎨 Рекомендации для графиков:")
if multi_style_monuments == 0:
    print("  ✅ Простые бар-чарты подойдут (один стиль на памятник)")
else:
    print("  ⚠️ Нужны stacked bar или pie chart (несколько стилей на памятник)")

if latin_styles > total_styles * 0.1:
    print("  ⚠️ Более 10% стилей на латинице — проверьте группировку перед визуализацией")
else:
    print("  ✅ Названия стилей однородны — можно строить графики без дополнительной очистки")

print("\n🚀 Готовы к построению визуализаций в Week 3!")

🚀 Запуск анализа стилевого разнообразия...

📊 ИНДЕКС СТИЛЕВОГО РАЗНООБРАЗИЯ ПО СТРАНАМ

🏆 Топ-10 стран по количеству уникальных стилей:


,unique_styles_count,shannon_entropy,total_monuments
country,,,
Испания,18,2.819834,67
Италия,15,3.525213,50
Франция,15,3.821928,20
Германия,11,2.771511,25
Чехия,10,1.396282,103
Польша,6,2.500000,8
Ватикан,5,2.251629,12
Иран,5,2.067031,14
Аргентина,4,2.000000,4



🧱 Топ-10 самых «однородных» стран (меньше всего стилей):


,unique_styles_count,shannon_entropy,total_monuments
country,,,
Арагонская корона,1,-0.0,1
Афганистан,1,-0.0,1
Бенин,1,-0.0,1
Великое княжество Литовское,1,-0.0,1
Венгрия,1,-0.0,2
Иерусалимское королевство,1,-0.0,1
Израиль,1,-0.0,1
Ирак,1,-0.0,1
Колумбия,1,-0.0,3



📈 Корреляция между числом памятников и разнообразием:
Коэффициент корреляции Пирсона: 0.757
💡 Интерпретация: чем ближе к 1, тем сильнее связь «больше памятников → больше стилей»

👻 АНАЛИЗ «СТИЛЕЙ-ПРИЗРАКОВ»

✨ Эксклюзивные стили (встречаются только в одной стране): 59
Топ-15 по частоте:


...


🏰 Страны с наибольшим числом эксклюзивных стилей:


,exclusive_styles_count
country,
Италия,9
Испания,8
Франция,7
Германия,5
Иран,4
Аргентина,2
Ватикан,2
Греция,2
Мексика,2



🔎 Примеры памятников с эксклюзивными стилями:


,URL,monument,country,style
3,http://www.wikidata.org/entity/Q20875264,Panteó Mundet,Испания,каталонский модерн
6,http://www.wikidata.org/entity/Q20875308,Sepulcre de la família Solà-Vinardell,Испания,каталонский модерн
24,http://www.wikidata.org/entity/Q126417669,Grave of Alvisi,Италия,стиль либерти
36,http://www.wikidata.org/entity/Q135579819,Kenotaph of Frederik I of Denmark,Германия,искусство Возрождения
47,http://www.wikidata.org/entity/Q132561980,Epitaph für Jürgen Wackerbarth und seine Frau ...,Германия,искусство Возрождения
48,http://www.wikidata.org/entity/Q132666517,Epitaph für die Familie Trotha,Германия,искусство Возрождения
49,http://www.wikidata.org/entity/Q134520839,Epitaph für Hans Christoph von Absberg,Германия,искусство Возрождения
50,http://www.wikidata.org/entity/Q134719249,Epitaph für Reichart und Anna Klieber,Германия,искусство Возрождения
53,http://www.wikidata.org/entity/Q134726930,"Epitaph für Johann Laurentius, Johann, Hierony...",Германия,искусство Возрождения
54,http://www.wikidata.org/entity/Q134729699,Epitaph für Karl von Lyra (Liere) alias von Ym...,Германия,искусство Возрождения



🔍 РАЗВЕДКА ДЛЯ ЗАДАНИЯ 3: ГРАФИКИ И ВИЗУАЛИЗАЦИЯ

📊 БЛОК А. Распределение числа стилей на памятник
----------------------------------------------------------------------
Распределение числа стилей на памятник:
style
1    277
2     50
3      5
4      4
6      1
Name: count, dtype: int64

🏛️ Памятников с 2+ стилями: 60 из 337
💡 Доля: 17.8%
⚠️ Внимание: есть памятники с несколькими стилями — это важно для визуализации!

----------------------------------------------------------------------
📊 БЛОК Б. Топ-10 пар стилей, встречающихся в одном памятнике
----------------------------------------------------------------------
Топ-10 пар стилей, встречающихся в одном памятнике:
  17x  модерн  +  стиль либерти
  14x  барокко  +  барокко
  8x  ар-деко  +  ар-деко
  6x  модерн  +  модерн
  5x  социалистический реализм  +  социалистический реализм
  5x  Anti-monuments in Mexico  +  Anti-monuments in Mexico
  5x  скульптура барокко  +  скульптура барокко
  5x  стиль либерти  +  стиль либерти
  4x  ит

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий [`python-ai-Evdokimova-Daria`](https://github.com/dasha3000/python-ai-Evdokimova-Daria) в Google Colab
- ✅ Прочитали CSV-файл `data/monuments_country_style.csv` в pandas DataFrame
- ✅ Очистили и переименовали столбцы:
  - `monument` (URL) → `URL` (сохранили для ссылок на Wikidata)
  - `monumentLabel` → `monument`, `countryLabel` → `country`, `styleLabel` → `style` (убрали суффикс `*Label`)
- ✅ Проверили структуру данных:
  - размер таблицы: **~400 строк × 4 столбца**
  - типы данных: все столбцы — строковые (категориальные)
  - первые строки и пропущенные значения
- ✅ Выполнили углублённый анализ:
  - 📊 **Индекс стилевого разнообразия**: посчитали количество уникальных стилей и индекс Шеннона для каждой из 40 стран
  - 👻 **«Стили-призраки»**: нашли эксклюзивные стили (встречаются только в одной стране) и редкие стили (1–3 вхождения)
  - 🌍 **Топ стран**: Чехия (103 памятника), Испания (67), Италия (50)
  - 🎨 **Топ стилей**: барокко (92), модерн (48), Anti-monuments in Mexico (23)

**Ключевые инсайты:**
> - Чехия лидирует не только по количеству памятников, но и по стилевому разнообразию
> - Мексика — «хранитель» уникальных стилей (например, «Anti-monuments in Mexico»)
> - Барокко — самый распространённый стиль, но его доминирование неравномерно по странам

Теперь у нас есть **аккуратный, проверенный DataFrame `df_monuments`**, с которым удобно работать дальше.

**В отдельном ноутбуке для следующей недели (Week 3) мы будем использовать эти данные для:**

- 🔍 **Продвинутой фильтрации**:
  - поиск памятников по стилю, стране или названию
  - комбинированные условия (например, «барокко в Италии»)
- 📊 **Группировки и агрегации**:
  - статистика по странам: сколько памятников, сколько стилей
  - сравнение регионов: Европа vs. Латинская Америка
- 🎨 **Визуализации**:
  - бар-чарты: топ стран и стилей
  - тепловая карта: страна × стиль
  - (опционально) интерактивная карта с `folium`
- 🌐 **Работа с Wikidata**:
  - использование столбца `URL` для получения дополнительных данных (координаты, год постройки, автор)

🗿 **Готовы превратить данные в историю?** Переходим к Week 3! ✨